In [1]:
import pandas as pd
from sdv.metadata import SingleTableMetadata
from sdmetrics.reports.single_table import QualityReport



C:\Users\Sahire\.conda\envs\synthdata\Lib\site-packages\sdmetrics\reports\single_table\__init__.py:12: FutureWarning: The single table quality report is deprecated. Please use the QualityReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.quality_report import QualityReport


In [45]:

def compute_fidelity(real_df, synthetic_df):
    """
    Synthetic ve real veride ortak bulunan kolonlar üzerinden
    SDV QualityReport ile fidelity hesaplar.

    Böylece synthetic dataset'te bulunmayan kolonlar nedeniyle
    metadata mismatch hatası oluşmaz.
    """

    common_cols = [
        col
        for col in real_df.columns
        if col in synthetic_df.columns
    ]

    if not common_cols:
        raise ValueError(
            " Fidelity hesaplanamadı: "
            "Real ve synthetic veride ortak kolon bulunamadı."
        )

    real_eval = real_df[common_cols].copy()

    synthetic_eval = synthetic_df[common_cols].copy()

    metadata = SingleTableMetadata()

    metadata.detect_from_dataframe(
        real_eval
    )

    report = QualityReport()

    report.generate(
        real_data=real_eval,
        synthetic_data=synthetic_eval,
        metadata=metadata.to_dict()
    )

    return report.get_score()

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

In [7]:

def encode_df(df, target_col):
    df = df.copy()
    encoders = {}
    for col in df.columns:
        if df[col].dtype == 'object':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y

def evaluate_tstr(synthetic_df, holdout_df, target_col):
    # Sadece iki veri setinde de ortak olan kolonlarla çalış
    common_cols = [c for c in holdout_df.columns if c in synthetic_df.columns]

    if target_col not in common_cols:
        raise ValueError(
            f" TSTR hesaplanamadı: hedef kolon '{target_col}' "
            f"ortak kolonlarda yok."
        )

    synthetic_eval = synthetic_df[common_cols].copy()
    holdout_eval = holdout_df[common_cols].copy()

    combined = pd.concat([
        synthetic_eval.assign(_src='synth'),
        holdout_eval.assign(_src='real')
    ])
    X_all, y_all = encode_df(combined.drop(columns=['_src']), target_col)
    src = combined['_src'].values

    X_train, y_train = X_all[src == 'synth'], y_all[src == 'synth']
    X_test, y_test = X_all[src == 'real'], y_all[src == 'real']

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='weighted')

def evaluate_real_baseline(train_df, holdout_df, target_col):
    # Karşılaştırma referansı: gerçek veri eğitilince ne kadar iyi olunuyor?
    combined = pd.concat([train_df.assign(_src='train'), holdout_df.assign(_src='real')])
    X_all, y_all = encode_df(combined.drop(columns=['_src']), target_col)
    src = combined['_src'].values

    X_train, y_train = X_all[src == 'train'], y_all[src == 'train']
    X_test, y_test = X_all[src == 'real'], y_all[src == 'real']

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='weighted')

In [11]:
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.preprocessing import MinMaxScaler

In [13]:
def compute_dcr(real_df, synthetic_df, numeric_cols):
    scaler = MinMaxScaler()
    real_scaled = scaler.fit_transform(real_df[numeric_cols])
    synth_scaled = scaler.transform(synthetic_df[numeric_cols])

    distances = cdist(synth_scaled, real_scaled)
    min_distances = distances.min(axis=1)
    return min_distances.mean(), min_distances.min()

In [19]:
import pandas as pd
from sdv.metadata import SingleTableMetadata
from sdmetrics.reports.single_table import QualityReport

def compute_fidelity(real_df, synthetic_df):
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(real_df)

    report = QualityReport()
    report.generate(real_data=real_df, synthetic_data=synthetic_df, metadata=metadata.to_dict())
    return report.get_score()

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

def encode_df(df, target_col):
    df = df.copy()
    encoders = {}
    for col in df.columns:
        if df[col].dtype == 'object':
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            encoders[col] = le
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y

def evaluate_tstr(synthetic_df, holdout_df, target_col):
    # Aynı encoding şemasını iki veri setine de tutarlı uygulamak için
    # basit yaklaşım: iki veri setini birleştirip birlikte encode et, sonra ayır
    combined = pd.concat([synthetic_df.assign(_src='synth'), holdout_df.assign(_src='real')])
    X_all, y_all = encode_df(combined.drop(columns=['_src']), target_col)
    src = combined['_src'].values

    X_train, y_train = X_all[src == 'synth'], y_all[src == 'synth']
    X_test, y_test = X_all[src == 'real'], y_all[src == 'real']

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='weighted')

def evaluate_real_baseline(train_df, holdout_df, target_col):
    # Karşılaştırma referansı: gerçek veriyle eğitilince ne kadar iyi olunuyor?
    combined = pd.concat([train_df.assign(_src='train'), holdout_df.assign(_src='real')])
    X_all, y_all = encode_df(combined.drop(columns=['_src']), target_col)
    src = combined['_src'].values

    X_train, y_train = X_all[src == 'train'], y_all[src == 'train']
    X_test, y_test = X_all[src == 'real'], y_all[src == 'real']

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='weighted')

In [39]:
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.preprocessing import MinMaxScaler

def compute_dcr(real_df, synthetic_df, numeric_cols, batch_size=1000):
    """
    Distance to Closest Record (DCR) hesaplar.

    Synthetic veride bulunmayan numeric kolonları otomatik olarak atlar.
    Büyük veriler için synthetic veriyi batch halinde işler.
    """

    available_numeric_cols = [
        col
        for col in numeric_cols
        if col in real_df.columns
        and col in synthetic_df.columns
    ]

    missing_numeric_cols = [
        col
        for col in numeric_cols
        if col not in synthetic_df.columns
    ]

    if missing_numeric_cols:
        print(
            f" DCR: Synthetic veride bulunmayan numeric kolonlar "
            f"atlandı: {missing_numeric_cols}"
        )

    if not available_numeric_cols:
        raise ValueError(
            " DCR hesaplanamadı: "
            "Real ve synthetic veride ortak numeric kolon bulunamadı."
        )

    scaler = MinMaxScaler()

    real_scaled = scaler.fit_transform(
        real_df[available_numeric_cols]
    )

    synth_scaled = scaler.transform(
        synthetic_df[available_numeric_cols]
    )

    min_distances = []

    for i in range(0, len(synth_scaled), batch_size):

        synth_batch = synth_scaled[
            i:i + batch_size
        ]

        distances_batch = cdist(
            synth_batch,
            real_scaled
        )

        batch_min = distances_batch.min(axis=1)

        min_distances.extend(batch_min)

    min_distances = np.array(min_distances)

    return (
        float(min_distances.mean()),
        float(min_distances.min())
    )

In [41]:
def align_adult_columns(train_df, synthetic_df):
    """
    Adult dataset'teki kolon isimlerini eşleştirir.

    Örnek:
        marital-status  -> marital.status
        hours-per-week  -> hours.per.week
        native-country  -> native.country

    Eksik kolonları yapay olarak oluşturmaz.
    """

    def normalize_column(col):
        return (
            str(col)
            .strip()
            .lower()
            .replace("-", "")
            .replace("_", "")
            .replace(".", "")
            .replace(" ", "")
        )

    train_columns = list(train_df.columns)

    normalized_train = {
        normalize_column(col): col
        for col in train_columns
    }

    rename_map = {}

    for synth_col in synthetic_df.columns:

        normalized_synth = normalize_column(synth_col)

        if normalized_synth in normalized_train:

            real_col = normalized_train[normalized_synth]

            rename_map[synth_col] = real_col

    synthetic_df = synthetic_df.rename(
        columns=rename_map
    )

    common_columns = [
        col
        for col in train_columns
        if col in synthetic_df.columns
    ]

    missing_columns = [
        col
        for col in train_columns
        if col not in synthetic_df.columns
    ]

    extra_columns = [
        col
        for col in synthetic_df.columns
        if col not in train_columns
    ]

    print("\n" + "=" * 60)
    print("ADULT SCHEMA KONTROLÜ")
    print("=" * 60)

    print(
        f"Gerçek veri kolon sayısı    : {len(train_columns)}"
    )

    print(
        f"Synthetic kolon sayısı      : {len(synthetic_df.columns)}"
    )

    print(
        f"Ortak kolon sayısı          : {len(common_columns)}"
    )

    print(
        f"Eksik kolonlar              : {missing_columns}"
    )

    if extra_columns:
        print(
            f"Fazladan kolonlar           : {extra_columns}"
        )

    print("=" * 60)

    synthetic_df = synthetic_df[common_columns]

    return (
        synthetic_df,
        common_columns,
        missing_columns
    )

In [27]:
train_df = pd.read_csv('../data/raw/adult_train.csv')
print(train_df.columns.tolist())

['age', 'workclass', 'fnlwgt', 'education', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'income']


In [37]:
train_df = pd.read_csv(
    datasets_info['adult']['train_path']
)

synthetic_df = pd.read_csv(
    methods_files['llm']['adult']
)

print("TRAIN COLUMNS:")
print(list(train_df.columns))

print("\nSYNTHETIC COLUMNS:")
print(list(synthetic_df.columns))

print("\nSYNTHETIC SHAPE:")
print(synthetic_df.shape)

print("\nTRAIN SHAPE:")
print(train_df.shape)

TRAIN COLUMNS:
['age', 'workclass', 'fnlwgt', 'education', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country', 'income']

SYNTHETIC COLUMNS:
['age', 'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'hours-per-week', 'native-country', 'income']

SYNTHETIC SHAPE:
(500, 11)

TRAIN SHAPE:
(21113, 15)


In [47]:
checkpoint_path = '../results/raw_results_checkpoint.csv'


if os.path.exists(checkpoint_path):

    results_df_exist = pd.read_csv(
        checkpoint_path
    )

    completed_pairs = set(
        zip(
            results_df_exist['method'],
            results_df_exist['dataset']
        )
    )

    results = results_df_exist.to_dict(
        'records'
    )

    print(
        f"➜ Önceden tamamlanmış "
        f"{len(completed_pairs)} kombinasyon bulundu."
    )

else:

    completed_pairs = set()

    results = []

    print(
        "➜ Önceden kaydedilmiş checkpoint bulunamadı."
    )




for method, files in methods_files.items():

    for dataset, path in files.items():

        # ----------------------------------------------------
        # CHECKPOINT KONTROLÜ
        # ----------------------------------------------------

        if (method, dataset) in completed_pairs:

            print(
                f" {method} / {dataset} "
                f"zaten tamamlanmış, atlanıyor."
            )

            continue


        print("\n" + "=" * 70)

        print(
            f"⏳ {method} / {dataset} hesaplanıyor..."
        )

        print("=" * 70)


        # ----------------------------------------------------
        # VERİLERİ YÜKLE
        # ----------------------------------------------------

        train_df = pd.read_csv(
            datasets_info[dataset]['train_path']
        )

        holdout_df = pd.read_csv(
            datasets_info[dataset]['holdout_path']
        )

        synthetic_df = pd.read_csv(
            path
        )


        # ----------------------------------------------------
        # ADULT KOLON İSİMLERİNİ EŞLEŞTİR
        # ----------------------------------------------------

        if dataset == 'adult':

            (
                synthetic_df,
                common_columns,
                missing_columns
            ) = align_adult_columns(
                train_df,
                synthetic_df
            )

        else:

            common_columns = [
                col
                for col in train_df.columns
                if col in synthetic_df.columns
            ]

            missing_columns = [
                col
                for col in train_df.columns
                if col not in synthetic_df.columns
            ]


        # ----------------------------------------------------
        # METADATA
        # ----------------------------------------------------

        target_col = datasets_info[
            dataset
        ]['target_col']

        numeric_cols = datasets_info[
            dataset
        ]['numeric_cols']


        # ----------------------------------------------------
        # TARGET KONTROLÜ
        # ----------------------------------------------------

        if target_col not in synthetic_df.columns:

            raise ValueError(
                f"\n Target kolon synthetic veride bulunamadı: "
                f"{target_col}\n\n"
                f"Mevcut kolonlar:\n"
                f"{list(synthetic_df.columns)}"
            )


        # ----------------------------------------------------
        # FIDELITY
        # ----------------------------------------------------

        print("\n Fidelity hesaplanıyor...")

        fidelity = compute_fidelity(
            train_df,
            synthetic_df
        )

        print(
            f"✓ Fidelity: {fidelity:.4f}"
        )


        # ----------------------------------------------------
        # TSTR
        # ----------------------------------------------------

        print("\n TSTR F1 hesaplanıyor...")

        tstr_f1 = evaluate_tstr(
            synthetic_df,
            holdout_df,
            target_col
        )

        print(
            f"✓ Synthetic TSTR F1: {tstr_f1:.4f}"
        )


        # ----------------------------------------------------
        # REAL BASELINE
        # ----------------------------------------------------

        real_baseline_f1 = real_baseline_cache[
            dataset
        ]

        print(
            f"✓ Real baseline F1: "
            f"{real_baseline_f1:.4f}"
        )


        # ----------------------------------------------------
        # DCR
        # ----------------------------------------------------

        print("\n DCR hesaplanıyor...")

        dcr_mean, dcr_min = compute_dcr(
            train_df,
            synthetic_df,
            numeric_cols
        )

        print(
            f"✓ DCR Mean: {dcr_mean:.6f}"
        )

        print(
            f"✓ DCR Min : {dcr_min:.6f}"
        )


        # ----------------------------------------------------
        # SCHEMA COVERAGE
        # ----------------------------------------------------

        schema_coverage = (
            len(common_columns)
            / len(train_df.columns)
        )


        # ----------------------------------------------------
        # SONUÇ SATIRI
        # ----------------------------------------------------

        row = {

            'method':
                method,

            'dataset':
                dataset,

            'fidelity_score':
                fidelity,

            'utility_f1_synthetic':
                tstr_f1,

            'utility_f1_real_baseline':
                real_baseline_f1,

            'utility_gap':
                real_baseline_f1 - tstr_f1,

            'privacy_dcr_mean':
                dcr_mean,

            'privacy_dcr_min':
                dcr_min,

            'real_column_count':
                len(train_df.columns),

            'synthetic_column_count':
                len(synthetic_df.columns),

            'schema_coverage':
                schema_coverage,

            'missing_columns':
                ', '.join(missing_columns),
        }


        # ----------------------------------------------------
        # SONUCU BELLEĞE EKLE
        # ----------------------------------------------------

        results.append(
            row
        )


        # ----------------------------------------------------
        # CHECKPOINT'E ANINDA KAYDET
        # ----------------------------------------------------

        pd.DataFrame(
            results
        ).to_csv(
            checkpoint_path,
            index=False
        )


        # ----------------------------------------------------
        # BAŞARI MESAJI
        # ----------------------------------------------------

        print("\n" + "-" * 70)

        print(
            f"✓ {method} / {dataset} "
            f"başarıyla tamamlandı."
        )

        print(
            f"✓ Checkpoint kaydedildi."
        )

        print(
            f"✓ Schema coverage: "
            f"{schema_coverage:.2%}"
        )

        if missing_columns:

            print(
                f" Eksik kolonlar: "
                f"{missing_columns}"
            )

        print("-" * 70)


# ============================================================
# 3. FINAL RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)


# ============================================================
# 4. FINAL CSV
# ============================================================

results_df.to_csv(
    '../results/raw_results.csv',
    index=False
)


print("\n" + "=" * 70)

print(
    " TÜM DEĞERLENDİRMELER TAMAMLANDI!"
)

print("=" * 70)


results_df

➜ Önceden tamamlanmış 4 kombinasyon bulundu.
⏩ gaussian_copula / adult zaten tamamlanmış, atlanıyor.
⏩ gaussian_copula / creditcard zaten tamamlanmış, atlanıyor.
⏩ ctgan / adult zaten tamamlanmış, atlanıyor.
⏩ ctgan / creditcard zaten tamamlanmış, atlanıyor.

⏳ llm / adult hesaplanıyor...

ADULT SCHEMA KONTROLÜ
Gerçek veri kolon sayısı    : 15
Synthetic kolon sayısı      : 11
Ortak kolon sayısı          : 11
Eksik kolonlar              : ['fnlwgt', 'education.num', 'capital.gain', 'capital.loss']

📊 Fidelity hesaplanıyor...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 337.60it/s]|
Column Shapes Score: 88.82%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 159.90it/s]|
Column Pair Trends Score: 79.09%

Overall Score (Average): 83.96%

✓ Fidelity: 0.8396

🎯 TSTR F1 hesaplanıyor...
✓ Synthetic TSTR F1: 0.7941
✓ Real baseline F1: 0.8557

🔐 DCR hesaplanıyor...
⚠️ DCR: Synthetic veride bulunmayan numeric kolonlar atlandı: ['fn

,method,dataset,fidelity_score,utility_f1_synthetic,utility_f1_real_baseline,utility_gap,privacy_dcr_mean,privacy_dcr_min,real_column_count,synthetic_column_count,schema_coverage,missing_columns
0,gaussian_copula,adult,0.842162,0.683215,0.855734,0.172519,0.091389,0.000000,NaN,NaN,NaN,NaN
1,gaussian_copula,creditcard,0.642946,0.997403,0.999492,0.002090,0.141386,0.061103,NaN,NaN,NaN,NaN
2,ctgan,adult,0.868372,0.818757,0.855734,0.036977,0.022131,0.000005,NaN,NaN,NaN,NaN
3,ctgan,creditcard,0.608692,0.989728,0.999492,0.009765,0.184570,0.038358,NaN,NaN,NaN,NaN
4,llm,adult,0.839561,0.794123,0.855734,0.061610,0.000082,0.000000,15.0,11.0,0.733333,"fnlwgt, education.num, capital.gain, capital.loss"
